# Load deps

In [ ]:
from dotenv import load_dotenv
load_dotenv()
from openai import OpenAI
from sqlitesearch import TextSearchIndex

from src.ingest import load_faq_data, build_index, build_sqlite_index
from src.rag_helper import RAGBase

# Load documents

In [ ]:
documents = load_faq_data()

# Index documents

In [ ]:
index = build_index(documents)

In [ ]:
sqlite_index = build_sqlite_index(
    [doc for doc in documents if doc["course"] == "llm-zoomcamp"],
    db_path="KB/llmzc_faq.db"
)

# Create LLM client

In [ ]:
openai_client = OpenAI()

# Load prompt components

In [ ]:
with open("templates/instructions.txt", "r") as file:
    instructions = file.read().strip()
# print(instructions)

In [ ]:
with open("templates/prompt_template.txt", "r") as file:
    prompt_template = file.read().strip()
# print(prompt_template)

# Create RAG instance

In [ ]:
assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
    prompt_template=prompt_template
)

# Run the rag pipeline

In [ ]:
answer = assistant.rag("I just discovered the course. Can I join now?")
print(answer)

# Run the rag with sqlitesearch

In [ ]:
sqlite_index = TextSearchIndex(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"],
    db_path="KB/llmzc_faq.db"
)

assistant = RAGBase(
    index=sqlite_index,
    llm_client=openai_client,
    instructions=instructions,
    prompt_template=prompt_template
)

answer = assistant.rag("I just discovered the course. Can I join now?")
print(answer)

In [ ]:
sqlite_index.close()